In [1]:
# Database Future Fixtures Analysis
# ==================================

import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

# Database connection
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def analyze_future_fixtures():
    """Analyze upcoming fixtures and available data"""
    
    conn = sqlite3.connect(db_path)
    
    print("🔮 FUTURE FIXTURES ANALYSIS")
    print("=" * 50)
    
    # Get current date for reference
    current_date = datetime.now().strftime('%Y-%m-%d')
    print(f"📅 Current date: {current_date}")
    
    # Query 1: Count upcoming fixtures by status
    upcoming_fixtures_query = """
    SELECT 
        status,
        COUNT(*) as fixture_count,
        MIN(starting_at) as earliest_match,
        MAX(starting_at) as latest_match
    FROM fixtures 
    WHERE starting_at > datetime('now')
    GROUP BY status
    ORDER BY fixture_count DESC
    """
    
    upcoming_by_status = pd.read_sql_query(upcoming_fixtures_query, conn)
    
    if not upcoming_by_status.empty:
        print(f"\n📊 UPCOMING FIXTURES BY STATUS:")
        print("-" * 40)
        total_upcoming = upcoming_by_status['fixture_count'].sum()
        print(f"Total upcoming fixtures: {total_upcoming:,}")
        print()
        for _, row in upcoming_by_status.iterrows():
            print(f"   {row['status']}: {row['fixture_count']:,} fixtures")
            print(f"      From: {row['earliest_match']}")
            print(f"      To: {row['latest_match']}")
            print()
    
    # Query 2: Upcoming fixtures by league
    upcoming_by_league_query = """
    SELECT 
        l.name as league_name,
        c.name as country,
        COUNT(f.id) as fixture_count,
        MIN(f.starting_at) as next_match,
        MAX(f.starting_at) as last_match
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    JOIN countries c ON l.country_id = c.id
    WHERE f.starting_at > datetime('now')
    GROUP BY l.id, l.name, c.name
    HAVING fixture_count > 0
    ORDER BY fixture_count DESC
    """
    
    upcoming_by_league = pd.read_sql_query(upcoming_by_league_query, conn)
    
    if not upcoming_by_league.empty:
        print(f"📊 UPCOMING FIXTURES BY LEAGUE (Top 10):")
        print("-" * 50)
        for i, (_, row) in enumerate(upcoming_by_league.head(10).iterrows(), 1):
            print(f"{i:2d}. {row['league_name']} ({row['country']})")
            print(f"    {row['fixture_count']:,} fixtures | Next: {row['next_match'][:10]}")
    
    # Query 3: Fixtures in next 30 days
    next_30_days_query = """
    SELECT 
        DATE(starting_at) as match_date,
        COUNT(*) as matches_per_day,
        GROUP_CONCAT(DISTINCT l.name) as leagues
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    WHERE f.starting_at BETWEEN datetime('now') AND datetime('now', '+30 days')
    GROUP BY DATE(starting_at)
    ORDER BY match_date
    """
    
    next_30_days = pd.read_sql_query(next_30_days_query, conn)
    
    if not next_30_days.empty:
        print(f"\n📅 FIXTURES IN NEXT 30 DAYS:")
        print("-" * 40)
        total_next_30 = next_30_days['matches_per_day'].sum()
        print(f"Total matches in next 30 days: {total_next_30:,}")
        print()
        print("Upcoming match schedule:")
        for _, row in next_30_days.head(10).iterrows():
            leagues_short = row['leagues'][:50] + "..." if len(row['leagues']) > 50 else row['leagues']
            print(f"   {row['match_date']}: {row['matches_per_day']:2d} matches ({leagues_short})")
    
    # Query 4: Check for available odds on upcoming fixtures
    upcoming_odds_query = """
    SELECT 
        COUNT(DISTINCT f.id) as fixtures_with_odds,
        COUNT(DISTINCT fo.bookmaker_name) as unique_bookmakers,
        COUNT(*) as total_odds_records
    FROM fixtures f
    JOIN fixture_odds fo ON f.id = fo.fixture_id
    WHERE f.starting_at > datetime('now')
    """
    
    upcoming_odds = pd.read_sql_query(upcoming_odds_query, conn)
    
    if not upcoming_odds.empty and upcoming_odds.iloc[0]['fixtures_with_odds'] > 0:
        print(f"\n💰 BETTING ODDS AVAILABILITY:")
        print("-" * 40)
        print(f"Fixtures with odds: {upcoming_odds.iloc[0]['fixtures_with_odds']:,}")
        print(f"Unique bookmakers: {upcoming_odds.iloc[0]['unique_bookmakers']}")
        print(f"Total odds records: {upcoming_odds.iloc[0]['total_odds_records']:,}")
    else:
        print(f"\n💰 BETTING ODDS AVAILABILITY:")
        print("-" * 40)
        print("❌ No odds data found for upcoming fixtures")
    
    # Query 5: Sample upcoming fixtures with team names
    sample_fixtures_query = """
    SELECT 
        f.starting_at,
        l.name as league,
        ht.name as home_team,
        at.name as away_team,
        f.status
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    JOIN teams ht ON f.home_team_id = ht.id
    JOIN teams at ON f.away_team_id = at.id
    WHERE f.starting_at > datetime('now')
    ORDER BY f.starting_at
    LIMIT 10
    """
    
    sample_fixtures = pd.read_sql_query(sample_fixtures_query, conn)
    
    if not sample_fixtures.empty:
        print(f"\n⚽ SAMPLE UPCOMING FIXTURES:")
        print("-" * 60)
        for _, row in sample_fixtures.iterrows():
            match_time = pd.to_datetime(row['starting_at']).strftime('%Y-%m-%d %H:%M')
            print(f"{match_time} | {row['league']}")
            print(f"   {row['home_team']} vs {row['away_team']} ({row['status']})")
    
    # Query 6: Check data freshness and last update
    data_freshness_query = """
    SELECT 
        'fixtures' as table_name,
        COUNT(*) as total_records,
        MAX(updated_at) as last_updated,
        COUNT(CASE WHEN starting_at > datetime('now') THEN 1 END) as future_records
    FROM fixtures
    UNION ALL
    SELECT 
        'fixture_odds' as table_name,
        COUNT(*) as total_records,
        MAX(updated_at) as last_updated,
        COUNT(CASE WHEN fixture_id IN (
            SELECT id FROM fixtures WHERE starting_at > datetime('now')
        ) THEN 1 END) as future_records
    FROM fixture_odds
    """
    
    data_freshness = pd.read_sql_query(data_freshness_query, conn)
    
    print(f"\n📊 DATA FRESHNESS:")
    print("-" * 40)
    for _, row in data_freshness.iterrows():
        print(f"{row['table_name']}:")
        print(f"   Total records: {row['total_records']:,}")
        print(f"   Future records: {row['future_records']:,}")
        print(f"   Last updated: {row['last_updated']}")
        print()
    
    conn.close()
    
    # Summary assessment
    total_upcoming = upcoming_by_status['fixture_count'].sum() if not upcoming_by_status.empty else 0
    
    print(f"🎯 PREDICTION READINESS ASSESSMENT:")
    print("-" * 50)
    
    if total_upcoming > 100:
        print(f"✅ EXCELLENT: {total_upcoming:,} upcoming fixtures available")
        print("   Ready for immediate predictions with optimized models!")
    elif total_upcoming > 20:
        print(f"✅ GOOD: {total_upcoming:,} upcoming fixtures available")
        print("   Sufficient data for testing optimized predictions")
    elif total_upcoming > 0:
        print(f"⚠️ LIMITED: Only {total_upcoming:,} upcoming fixtures")
        print("   Can test predictions but limited sample size")
    else:
        print(f"❌ NO UPCOMING FIXTURES: Database may need updating")
        print("   Wait for May 30th data refresh")
    
    # Check if we have odds
    odds_available = upcoming_odds.iloc[0]['fixtures_with_odds'] if not upcoming_odds.empty else 0
    
    if odds_available > 0:
        print(f"💰 BETTING READY: {odds_available:,} fixtures have odds data")
        print("   Can perform value betting analysis immediately!")
    else:
        print(f"💰 ODDS NEEDED: No pre-match odds available")
        print("   Will need to fetch odds data for value betting")
    
    return {
        'total_upcoming_fixtures': total_upcoming,
        'fixtures_with_odds': odds_available,
        'leagues_with_upcoming': len(upcoming_by_league) if not upcoming_by_league.empty else 0,
        'next_30_days_matches': next_30_days['matches_per_day'].sum() if not next_30_days.empty else 0
    }

# Run the analysis
future_fixtures_summary = analyze_future_fixtures()

print(f"\n" + "="*60)
print("📋 SUMMARY:")
print(f"   🔮 Total upcoming fixtures: {future_fixtures_summary['total_upcoming_fixtures']:,}")
print(f"   💰 Fixtures with odds: {future_fixtures_summary['fixtures_with_odds']:,}")
print(f"   🏆 Active leagues: {future_fixtures_summary['leagues_with_upcoming']}")
print(f"   📅 Next 30 days: {future_fixtures_summary['next_30_days_matches']:,} matches")

if future_fixtures_summary['total_upcoming_fixtures'] > 50:
    print(f"\n🚀 RECOMMENDATION: Proceed with ensemble optimization!")
    print("   You have sufficient upcoming fixtures to test optimized predictions")
else:
    print(f"\n⏳ RECOMMENDATION: Focus on optimization, wait for more fixture data")
print("="*60)

🔮 FUTURE FIXTURES ANALYSIS
📅 Current date: 2025-05-27

📊 UPCOMING FIXTURES BY STATUS:
----------------------------------------
Total upcoming fixtures: 349

   None: 349 fixtures
      From: 2025-05-28 17:00:00
      To: 2025-11-30 16:00:00

📊 UPCOMING FIXTURES BY LEAGUE (Top 10):
--------------------------------------------------
 1. Eliteserien (Norway)
    173 fixtures | Next: 2025-05-28
 2. Allsvenskan (Sweden)
    151 fixtures | Next: 2025-05-29
 3. La Liga 2 (Spain)
    11 fixtures | Next: 2025-06-01
 4. Super Lig (Turkey)
    9 fixtures | Next: 2025-06-01
 5. Admiral Bundesliga (Austria)
    2 fixtures | Next: 2025-05-29
 6. Serie B (Italy)
    2 fixtures | Next: 2025-05-29
 7. Superliga (Denmark)
    1 fixtures | Next: 2025-05-31

📅 FIXTURES IN NEXT 30 DAYS:
----------------------------------------
Total matches in next 30 days: 53

Upcoming match schedule:
   2025-05-28:  2 matches (Eliteserien)
   2025-05-29:  4 matches (Allsvenskan,Eliteserien,Serie B,Admiral Bundesliga)
   

This shows we have more than enough data to make the predictions, but i first need to get the odds for those fixtures, as they are missing.

In [4]:
# SportMonks API v3 Diagnostic & Correct Odds Fetching
# ===================================================

import requests
import sqlite3
import pandas as pd
import time
from datetime import datetime
import json

# SportMonks API v3 Diagnostic & Correct Odds Fetching
# ===================================================

import requests
import sqlite3
import pandas as pd
import time
from datetime import datetime
import json

# SportMonks API Configuration
API_TOKEN = "PgeMnb1Y71v04KzxFBpKQmm2sxsyWihIRNXSvDoYUz6ZuDOY3h1lLnmKamH1"
BASE_URL = "https://api.sportmonks.com/v3/football"
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def setup_api_session():
    """Setup requests session with proper headers for SportMonks API v3"""
    session = requests.Session()
    session.headers.update({
        'Authorization': f'{API_TOKEN}',  # SportMonks uses direct token
        'Accept': 'application/json'
    })
    return session

def test_api_authentication():
    """Test basic API authentication and subscription access"""
    
    print("🔐 TESTING API AUTHENTICATION")
    print("=" * 50)
    
    session = setup_api_session()
    
    # Test basic endpoint
    test_url = f"{BASE_URL}/leagues"
    
    try:
        response = session.get(test_url, timeout=10)
        
        print(f"📡 Test URL: {test_url}")
        print(f"📊 Status Code: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"✅ Authentication successful!")
            print(f"📋 Available leagues: {len(data.get('data', []))}")
            return True
        elif response.status_code == 401:
            print(f"❌ Authentication failed - Invalid API token")
            return False
        elif response.status_code == 403:
            print(f"❌ Access forbidden - Check subscription permissions")
            return False
        else:
            print(f"❌ API Error: {response.status_code}")
            print(f"Response: {response.text[:200]}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Connection error: {e}")
        return False

def explore_available_endpoints():
    """Explore what endpoints are available with the subscription"""
    
    print("\n🔍 EXPLORING AVAILABLE ENDPOINTS")
    print("=" * 50)
    
    session = setup_api_session()
    
    # Test various endpoints to see what's available
    endpoints_to_test = [
        ("Leagues", "/leagues"),
        ("Markets", "/markets"),
        ("Bookmakers", "/bookmakers"),
        ("Odds Live", "/odds/live"),
        ("Odds Pre-match", "/odds/pre-match"),
    ]
    
    available_endpoints = []
    
    for name, endpoint in endpoints_to_test:
        url = f"{BASE_URL}{endpoint}"
        try:
            response = session.get(url, timeout=10)
            print(f"📊 {name:<15} | {response.status_code} | {url}")
            
            if response.status_code == 200:
                available_endpoints.append((name, endpoint))
                # Show sample data structure
                data = response.json()
                if 'data' in data and len(data['data']) > 0:
                    sample = data['data'][0]
                    print(f"   Sample keys: {list(sample.keys())[:5]}")
            elif response.status_code == 403:
                print(f"   ❌ Not included in subscription")
            elif response.status_code == 404:
                print(f"   ❌ Endpoint not found")
                
        except Exception as e:
            print(f"   ❌ Error: {e}")
        
        time.sleep(0.5)  # Rate limiting
    
    return available_endpoints

def test_odds_endpoints_specifically():
    """Test different odds endpoint variations"""
    
    print("\n🎲 TESTING ODDS ENDPOINTS")
    print("=" * 50)
    
    session = setup_api_session()
    
    # Get a sample fixture ID from database
    conn = sqlite3.connect(db_path)
    sample_fixture_query = """
    SELECT id as fixture_id, starting_at, 
           (SELECT name FROM teams WHERE id = home_team_id) as home_team,
           (SELECT name FROM teams WHERE id = away_team_id) as away_team
    FROM fixtures 
    WHERE starting_at > datetime('now')
    ORDER BY starting_at
    LIMIT 3
    """
    
    sample_fixtures = pd.read_sql_query(sample_fixture_query, conn)
    conn.close()
    
    if sample_fixtures.empty:
        print("❌ No upcoming fixtures found for testing")
        return
    
    sample_fixture_id = sample_fixtures.iloc[0]['fixture_id']
    print(f"🎯 Testing with fixture ID: {sample_fixture_id}")
    print(f"   Match: {sample_fixtures.iloc[0]['home_team']} vs {sample_fixtures.iloc[0]['away_team']}")
    
    # Test different odds endpoint variations
    odds_endpoints = [
        f"/odds/pre-match/fixtures/{sample_fixture_id}",
        f"/fixtures/{sample_fixture_id}/odds",
        f"/odds/pre-match?fixture_id={sample_fixture_id}",
        f"/fixtures/{sample_fixture_id}?include=odds",
        f"/odds/fixtures/{sample_fixture_id}",
    ]
    
    for endpoint in odds_endpoints:
        url = f"{BASE_URL}{endpoint}"
        try:
            print(f"\n📊 Testing: {endpoint}")
            response = session.get(url, timeout=10)
            print(f"   Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                print(f"   ✅ SUCCESS! Found odds data")
                
                # Analyze structure
                if 'data' in data:
                    odds_data = data['data']
                    if isinstance(odds_data, list) and len(odds_data) > 0:
                        print(f"   📋 Odds records: {len(odds_data)}")
                        print(f"   🔑 Sample keys: {list(odds_data[0].keys())}")
                    elif isinstance(odds_data, dict):
                        print(f"   📋 Single odds object")
                        print(f"   🔑 Keys: {list(odds_data.keys())}")
                
                # This is our working endpoint!
                return endpoint, url
                
            elif response.status_code == 404:
                print(f"   ❌ Endpoint not found")
            elif response.status_code == 403:
                print(f"   ❌ Access denied - check subscription")
            else:
                print(f"   ❌ Error {response.status_code}: {response.text[:100]}")
                
        except Exception as e:
            print(f"   ❌ Request failed: {e}")
        
        time.sleep(1)  # Rate limiting
    
    return None, None

def test_fixture_with_includes():
    """Test fetching fixture data with includes parameter"""
    
    print("\n🔧 TESTING FIXTURE WITH INCLUDES")
    print("=" * 50)
    
    session = setup_api_session()
    
    # Get sample fixture
    conn = sqlite3.connect(db_path)
    sample_fixture_query = """
    SELECT id as fixture_id
    FROM fixtures 
    WHERE starting_at > datetime('now')
    ORDER BY starting_at
    LIMIT 1
    """
    
    result = pd.read_sql_query(sample_fixture_query, conn)
    conn.close()
    
    if result.empty:
        print("❌ No upcoming fixtures for testing")
        return
    
    fixture_id = result.iloc[0]['fixture_id']
    
    # Test different include parameters
    include_options = [
        "odds",
        "bookmakers",
        "markets", 
        "odds.bookmaker",
        "odds.market",
        "odds.bookmaker,odds.market"
    ]
    
    for include_param in include_options:
        url = f"{BASE_URL}/fixtures/{fixture_id}"
        params = {'include': include_param}
        
        try:
            print(f"\n📊 Testing include: {include_param}")
            response = session.get(url, params=params, timeout=10)
            print(f"   Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                
                if 'data' in data:
                    fixture_data = data['data']
                    print(f"   ✅ SUCCESS!")
                    print(f"   🔑 Available keys: {list(fixture_data.keys())}")
                    
                    # Check for odds-related data
                    for key in fixture_data.keys():
                        if 'odds' in key.lower() or 'market' in key.lower() or 'book' in key.lower():
                            print(f"   🎯 Found: {key}")
                            if isinstance(fixture_data[key], list):
                                print(f"      Count: {len(fixture_data[key])}")
                            
            else:
                print(f"   ❌ Error: {response.status_code}")
                
        except Exception as e:
            print(f"   ❌ Request failed: {e}")
        
        time.sleep(1)

def check_subscription_details():
    """Check what data is available with current subscription"""
    
    print("\n📋 CHECKING SUBSCRIPTION DETAILS")
    print("=" * 50)
    
    session = setup_api_session()
    
    # Test core endpoints that should work with basic subscription
    test_endpoints = [
        ("/leagues", "Leagues"),
        ("/seasons", "Seasons"), 
        ("/fixtures", "Fixtures"),
        ("/teams", "Teams"),
        ("/players", "Players"),
        ("/markets", "Markets"),
        ("/bookmakers", "Bookmakers")
    ]
    
    working_endpoints = []
    
    for endpoint, name in test_endpoints:
        url = f"{BASE_URL}{endpoint}"
        try:
            response = session.get(url, timeout=10)
            status = "✅" if response.status_code == 200 else "❌"
            print(f"{status} {name:<12} | {response.status_code}")
            
            if response.status_code == 200:
                working_endpoints.append(endpoint)
                
        except Exception as e:
            print(f"❌ {name:<12} | Error: {e}")
        
        time.sleep(0.5)
    
    return working_endpoints

def main_diagnostic():
    """Run complete diagnostic"""
    
    print("🔬 SPORTMONKS API v3 COMPREHENSIVE DIAGNOSTIC")
    print("=" * 70)
    
    # Step 1: Test authentication
    if not test_api_authentication():
        print("\n❌ Authentication failed. Please check your API token.")
        return
    
    # Step 2: Check subscription access
    working_endpoints = check_subscription_details()
    
    # Step 3: Explore available endpoints
    available_endpoints = explore_available_endpoints()
    
    # Step 4: Test odds-specific endpoints
    working_odds_endpoint, working_odds_url = test_odds_endpoints_specifically()
    
    # Step 5: Test fixture includes
    test_fixture_with_includes()
    
    # Summary
    print(f"\n🎯 DIAGNOSTIC SUMMARY")
    print("=" * 50)
    print(f"✅ Authentication: Working")
    print(f"📊 Working endpoints: {len(working_endpoints)}")
    
    if working_odds_endpoint:
        print(f"🎲 Odds endpoint found: {working_odds_endpoint}")
        print(f"🚀 Ready to fetch odds data!")
    else:
        print(f"❌ No working odds endpoint found")
        print(f"💡 Odds data might not be included in Basic plan")
        print(f"   or requires different endpoint structure")

# Run the diagnostic
if __name__ == "__main__":
    main_diagnostic()

🔬 SPORTMONKS API v3 COMPREHENSIVE DIAGNOSTIC
🔐 TESTING API AUTHENTICATION
📡 Test URL: https://api.sportmonks.com/v3/football/leagues
📊 Status Code: 200
✅ Authentication successful!
📋 Available leagues: 25

📋 CHECKING SUBSCRIPTION DETAILS
✅ Leagues      | 200
✅ Seasons      | 200
✅ Fixtures     | 200
✅ Teams        | 200
✅ Players      | 200
❌ Markets      | 404
❌ Bookmakers   | 404

🔍 EXPLORING AVAILABLE ENDPOINTS
📊 Leagues         | 200 | https://api.sportmonks.com/v3/football/leagues
   Sample keys: ['id', 'sport_id', 'country_id', 'name', 'active']
📊 Markets         | 404 | https://api.sportmonks.com/v3/football/markets
   ❌ Endpoint not found
📊 Bookmakers      | 404 | https://api.sportmonks.com/v3/football/bookmakers
   ❌ Endpoint not found
📊 Odds Live       | 404 | https://api.sportmonks.com/v3/football/odds/live
   ❌ Endpoint not found
📊 Odds Pre-match  | 200 | https://api.sportmonks.com/v3/football/odds/pre-match
   Sample keys: ['id', 'fixture_id', 'market_id', 'bookmaker_id', 

In [16]:
# Working SportMonks Odds Fetcher - Streamlined Version
# =====================================================

import requests
import sqlite3
import pandas as pd
import time
from datetime import datetime
import json
from tqdm import tqdm

# SportMonks API Configuration
API_TOKEN = "PgeMnb1Y71v04KzxFBpKQmm2sxsyWihIRNXSvDoYUz6ZuDOY3h1lLnmKamH1"
BASE_URL = "https://api.sportmonks.com/v3/football"
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def setup_session():
    """Setup requests session with proper headers"""
    session = requests.Session()
    session.headers.update({
        'Authorization': f'{API_TOKEN}',
        'Accept': 'application/json'
    })
    return session

def get_fixtures_needing_odds(limit=None):
    """Get upcoming fixtures that need odds data"""
    
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT DISTINCT
        f.id as fixture_id,
        f.starting_at,
        f.league_id,
        l.name as league_name,
        ht.name as home_team,
        at.name as away_team
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    JOIN teams ht ON f.home_team_id = ht.id
    JOIN teams at ON f.away_team_id = at.id
    WHERE f.starting_at > datetime('now')
    AND f.id NOT IN (
        SELECT DISTINCT fixture_id 
        FROM fixture_odds 
        WHERE fixture_id IS NOT NULL
    )
    ORDER BY f.starting_at
    """
    
    if limit:
        query += f" LIMIT {limit}"
    
    fixtures_df = pd.read_sql_query(query, conn)
    conn.close()
    
    print(f"📊 Found {len(fixtures_df)} fixtures needing odds data")
    return fixtures_df

def fetch_fixture_odds(session, fixture_id):
    """Fetch odds for a single fixture using the working endpoint"""
    
    url = f"{BASE_URL}/odds/pre-match/fixtures/{fixture_id}"
    
    try:
        response = session.get(url, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            odds_data = data.get('data', [])
            return odds_data
            
        elif response.status_code == 429:
            print(f"      ⏳ Rate limited, waiting...")
            time.sleep(60)
            return None  # Will retry
            
        elif response.status_code == 404:
            print(f"      ⚠️ No odds for fixture {fixture_id}")
            return []
            
        else:
            print(f"      ❌ Error {response.status_code}")
            return []
            
    except Exception as e:
        print(f"      ❌ Request failed: {str(e)[:100]}")
        return []

def safe_float_convert(value, default=None):
    """Safely convert value to float, handling percentages and other formats"""
    if value is None or value == '':
        return default
    
    try:
        # Handle percentage strings like '51.02%'
        if isinstance(value, str) and '%' in value:
            return float(value.replace('%', '')) / 100.0
        
        # Handle regular numbers
        return float(value)
    except (ValueError, TypeError):
        return default

def process_odds_data(odds_data, fixture_id):
    """Process and structure odds data for database insertion"""
    
    processed_odds = []
    
    for odds in odds_data:
        processed_odds.append({
            'id': odds.get('id'),
            'fixture_id': fixture_id,
            'market_id': odds.get('market_id'),
            'bookmaker_id': odds.get('bookmaker_id'),
            'bookmaker_name': odds.get('name', 'Unknown'),
            'market_name': odds.get('market_description', 'Unknown'),
            'odds_label': odds.get('label', ''),
            'odds_value': safe_float_convert(odds.get('value'), None),
            'probability': safe_float_convert(odds.get('probability'), None),
            'fractional': odds.get('fractional', ''),
            'american': odds.get('american', ''),
            'decimal_3': odds.get('dp3', ''),
            'handicap': safe_float_convert(odds.get('handicap'), None),
            'total': safe_float_convert(odds.get('total'), None),
            'sort_order': safe_float_convert(odds.get('sort_order'), 0),
            'winning': odds.get('winning'),
            'stopped': odds.get('stopped', False),
            'participants': odds.get('participants', ''),
            'latest_bookmaker_update': odds.get('latest_bookmaker_update', ''),
            'is_winning': None,
            'updated_at': datetime.now().isoformat()
        })
    
    return processed_odds

def insert_odds_batch(odds_batch):
    """Insert batch of odds into database with proper handling"""
    
    if not odds_batch:
        return 0
    
    conn = sqlite3.connect(db_path)
    
    try:
        # Convert to DataFrame and ensure updated_at is not None
        df = pd.DataFrame(odds_batch)
        
        # Ensure updated_at is always set
        if 'updated_at' not in df.columns or df['updated_at'].isna().any():
            df['updated_at'] = datetime.now().isoformat()
        
        # Handle potential None/NaN values for NOT NULL columns
        df['fixture_id'] = df['fixture_id'].fillna(0).astype(int)
        df['bookmaker_id'] = df['bookmaker_id'].fillna(0).astype(int)  
        df['market_id'] = df['market_id'].fillna(0).astype(int)
        
        # Insert using pandas
        df.to_sql('fixture_odds', conn, if_exists='append', index=False, method='multi')
        count = len(df)
        conn.close()
        return count
        
    except Exception as e:
        print(f"      ❌ Database error: {e}")
        conn.close()
        return 0

def fetch_all_odds(test_mode=False, test_limit=5):
    """Main function to fetch odds for all fixtures"""
    
    print("🎲 FETCHING SPORTMONKS ODDS DATA")
    print("=" * 50)
    
    # Get fixtures
    limit = test_limit if test_mode else None
    fixtures_df = get_fixtures_needing_odds(limit)
    
    if len(fixtures_df) == 0:
        print("✅ All fixtures already have odds!")
        return
    
    # Setup
    session = setup_session()
    odds_batch = []
    stats = {
        'successful': 0,
        'failed': 0,
        'total_odds': 0
    }
    
    print(f"🔄 Processing {len(fixtures_df)} fixtures...")
    if test_mode:
        print("   🧪 TEST MODE - Processing limited fixtures")
    
    # Process each fixture
    for i, (_, fixture) in enumerate(fixtures_df.iterrows(), 1):
        fixture_id = fixture['fixture_id']
        match_info = f"{fixture['home_team']} vs {fixture['away_team']}"
        
        print(f"\n{i}/{len(fixtures_df)} | {fixture['league_name']}")
        print(f"   {match_info}")
        print(f"   Fixture ID: {fixture_id}")
        
        # Fetch odds
        odds_data = fetch_fixture_odds(session, fixture_id)
        
        if odds_data is None:  # Rate limited, retry once
            time.sleep(60)
            odds_data = fetch_fixture_odds(session, fixture_id)
        
        if odds_data and len(odds_data) > 0:
            # Process and collect odds
            processed_odds = process_odds_data(odds_data, fixture_id)
            odds_batch.extend(processed_odds)
            stats['successful'] += 1
            stats['total_odds'] += len(processed_odds)
            
            print(f"   ✅ {len(processed_odds):,} odds fetched")
            
        else:
            stats['failed'] += 1
            print(f"   ❌ No odds available")
        
        # Batch insert every 1000 odds or at end
        if len(odds_batch) >= 1000 or i == len(fixtures_df):
            if odds_batch:
                inserted = insert_odds_batch(odds_batch)
                print(f"   💾 Inserted {inserted:,} odds to database")
                odds_batch = []
        
        # Rate limiting
        time.sleep(1.2)
    
    # Summary
    print(f"\n🎉 ODDS FETCHING COMPLETE!")
    print("=" * 40)
    print(f"   ✅ Successful: {stats['successful']}")
    print(f"   ❌ Failed: {stats['failed']}")
    print(f"   📊 Total odds: {stats['total_odds']:,}")
    
    if stats['successful'] > 0:
        avg_odds = stats['total_odds'] / stats['successful']
        print(f"   📈 Avg odds per fixture: {avg_odds:.0f}")
    
    return stats

def check_odds_coverage():
    """Check current odds coverage in database"""
    
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT 
        COUNT(DISTINCT f.id) as total_upcoming_fixtures,
        COUNT(DISTINCT fo.fixture_id) as fixtures_with_odds,
        ROUND(COUNT(DISTINCT fo.fixture_id) * 100.0 / COUNT(DISTINCT f.id), 1) as coverage_pct,
        COUNT(*) as total_odds_records
    FROM fixtures f
    LEFT JOIN fixture_odds fo ON f.id = fo.fixture_id
    WHERE f.starting_at > datetime('now')
    """
    
    result = pd.read_sql_query(query, conn)
    conn.close()
    
    print("📊 CURRENT ODDS COVERAGE")
    print("=" * 30)
    print(f"Total upcoming fixtures: {result.iloc[0]['total_upcoming_fixtures']:,}")
    print(f"Fixtures with odds: {result.iloc[0]['fixtures_with_odds']:,}")
    print(f"Coverage: {result.iloc[0]['coverage_pct']}%")
    print(f"Total odds records: {result.iloc[0]['total_odds_records']:,}")
    
    return result.iloc[0].to_dict()

# Test function for small batch
def test_odds_fetching():
    """Test with 5 fixtures first"""
    print("🧪 TESTING ODDS FETCHING")
    return fetch_all_odds(test_mode=True, test_limit=5)

# Full fetch function
def fetch_all_odds_full():
    """Fetch odds for all fixtures"""
    return fetch_all_odds(test_mode=False)

# Quick usage guide
if __name__ == "__main__":
    print("🎲 SPORTMONKS ODDS FETCHER")
    print("=" * 40)
    print("Available functions:")
    print("1. check_odds_coverage()   - Check current coverage")  
    print("2. test_odds_fetching()    - Test with 5 fixtures")
    print("3. fetch_all_odds_full()   - Fetch all odds")
    print("\nRecommended order:")
    print("check_odds_coverage() -> test_odds_fetching() -> fetch_all_odds_full()")

🎲 SPORTMONKS ODDS FETCHER
Available functions:
1. check_odds_coverage()   - Check current coverage
2. test_odds_fetching()    - Test with 5 fixtures
3. fetch_all_odds_full()   - Fetch all odds

Recommended order:
check_odds_coverage() -> test_odds_fetching() -> fetch_all_odds_full()


In [17]:
# Step 1: Check current odds coverage
print("STEP 1: Checking current odds coverage...")
coverage = check_odds_coverage()

STEP 1: Checking current odds coverage...
📊 CURRENT ODDS COVERAGE
Total upcoming fixtures: 349.0
Fixtures with odds: 0.0
Coverage: 0.0%
Total odds records: 349.0


In [18]:
# Step 2: Test again with fixed insertion
print("STEP 2: Testing with fixed insertion...")
test_results = test_odds_fetching()

STEP 2: Testing with fixed insertion...
🧪 TESTING ODDS FETCHING
🎲 FETCHING SPORTMONKS ODDS DATA
📊 Found 5 fixtures needing odds data
🔄 Processing 5 fixtures...
   🧪 TEST MODE - Processing limited fixtures

1/5 | Eliteserien
   Fredrikstad vs Rosenborg
   Fixture ID: 19354690
   ✅ 2,752 odds fetched
      ❌ Database error: UNIQUE constraint failed: fixture_odds.fixture_id, fixture_odds.bookmaker_id, fixture_odds.market_id, fixture_odds.odds_label
   💾 Inserted 0 odds to database

2/5 | Eliteserien
   Bodø / Glimt vs Viking
   Fixture ID: 19354697
   ✅ 2,967 odds fetched
      ❌ Database error: UNIQUE constraint failed: fixture_odds.fixture_id, fixture_odds.bookmaker_id, fixture_odds.market_id, fixture_odds.odds_label
   💾 Inserted 0 odds to database

3/5 | Admiral Bundesliga
   TBC vs SK Rapid
   Fixture ID: 19419846
   ❌ No odds available

4/5 | Allsvenskan
   Brommapojkarna vs Djurgården
   Fixture ID: 19347728
   ✅ 2,801 odds fetched
      ❌ Database error: UNIQUE constraint failed: 

now we see some issues regarding the fetch of the data

In [12]:
# Database Schema Fix for Odds Fetching
# =====================================

import sqlite3
import pandas as pd

db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def check_fixture_odds_schema():
    """Check current fixture_odds table structure"""
    
    conn = sqlite3.connect(db_path)
    
    # Get table info
    cursor = conn.cursor()
    cursor.execute("PRAGMA table_info(fixture_odds)")
    columns = cursor.fetchall()
    
    print("📊 CURRENT FIXTURE_ODDS TABLE SCHEMA:")
    print("=" * 50)
    for col in columns:
        print(f"   {col[1]:<25} | {col[2]:<10} | Nullable: {not col[3]}")
    
    # Check if table exists and has data
    cursor.execute("SELECT COUNT(*) FROM fixture_odds")
    count = cursor.fetchone()[0]
    print(f"\n📈 Current records: {count:,}")
    
    conn.close()
    return [col[1] for col in columns]  # Return column names

def add_missing_columns():
    """Add missing columns to fixture_odds table"""
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # List of columns we need that might be missing
    required_columns = [
        ('fractional', 'TEXT'),
        ('american', 'TEXT'), 
        ('decimal_3', 'TEXT'),
        ('handicap', 'REAL'),
        ('total', 'REAL'),
        ('sort_order', 'INTEGER'),
        ('winning', 'BOOLEAN'),
        ('stopped', 'BOOLEAN'),
        ('participants', 'TEXT'),
        ('latest_bookmaker_update', 'TEXT')
    ]
    
    print("🔧 ADDING MISSING COLUMNS:")
    print("=" * 40)
    
    for col_name, col_type in required_columns:
        try:
            cursor.execute(f"ALTER TABLE fixture_odds ADD COLUMN {col_name} {col_type}")
            print(f"   ✅ Added: {col_name} ({col_type})")
        except sqlite3.OperationalError as e:
            if "duplicate column name" in str(e):
                print(f"   ⚠️ Exists: {col_name}")
            else:
                print(f"   ❌ Error adding {col_name}: {e}")
    
    conn.commit()
    conn.close()
    
    print("\n✅ Schema update complete!")

def create_simplified_odds_processor():
    """Create a version that only uses existing columns"""
    
    current_columns = check_fixture_odds_schema()
    
    print(f"\n🔧 CREATING SIMPLIFIED PROCESSOR FOR EXISTING SCHEMA")
    print("=" * 60)
    
    # Show which columns we can use
    standard_columns = [
        'id', 'fixture_id', 'market_id', 'bookmaker_id', 
        'bookmaker_name', 'market_name', 'odds_label', 
        'odds_value', 'probability', 'is_winning', 'updated_at'
    ]
    
    available_columns = [col for col in standard_columns if col in current_columns]
    missing_columns = [col for col in standard_columns if col not in current_columns]
    
    print("✅ Available columns:")
    for col in available_columns:
        print(f"   - {col}")
    
    if missing_columns:
        print("\n❌ Missing standard columns:")
        for col in missing_columns:
            print(f"   - {col}")
    
    return available_columns

def fix_database_schema():
    """Main function to fix the database schema"""
    
    print("🗄️ FIXING DATABASE SCHEMA FOR ODDS")
    print("=" * 50)
    
    # Step 1: Check current schema
    current_columns = check_fixture_odds_schema()
    
    # Step 2: Try to add missing columns
    print(f"\n🔧 Attempting to add missing columns...")
    add_missing_columns()
    
    # Step 3: Verify the fix
    print(f"\n✅ VERIFICATION:")
    final_columns = check_fixture_odds_schema()
    
    return final_columns

# Alternative: Create a minimal processor that works with basic columns
def create_minimal_odds_processor():
    """Create processor that works with minimal column set"""
    
    code = '''
def process_odds_data_minimal(odds_data, fixture_id):
    """Minimal odds processor - only essential columns"""
    
    processed_odds = []
    
    for odds in odds_data:
        processed_odds.append({
            'id': odds.get('id'),
            'fixture_id': fixture_id,
            'market_id': odds.get('market_id'),
            'bookmaker_id': odds.get('bookmaker_id'),
            'bookmaker_name': odds.get('name', 'Unknown'),
            'market_name': odds.get('market_description', 'Unknown'),
            'odds_label': odds.get('label', ''),
            'odds_value': safe_float_convert(odds.get('value'), None),
            'probability': safe_float_convert(odds.get('probability'), None),
            'is_winning': None,
            'updated_at': datetime.now().isoformat()
        })
    
    return processed_odds
'''
    
    print("📝 MINIMAL ODDS PROCESSOR CODE:")
    print("=" * 40)
    print(code)
    
    return code

# Run the fix
if __name__ == "__main__":
    print("🗄️ DATABASE SCHEMA DIAGNOSTIC AND FIX")
    print("=" * 50)
    
    # Option 1: Try to fix schema
    print("OPTION 1: Fix database schema")
    final_columns = fix_database_schema()
    
    # Option 2: Create minimal processor  
    print(f"\nOPTION 2: Use minimal processor")
    minimal_code = create_minimal_odds_processor()
    
    print(f"\n🎯 RECOMMENDATION:")
    if len(final_columns) >= 10:
        print("✅ Schema fixed! You can proceed with the original code.")
    else:
        print("⚠️ Use the minimal processor above to avoid column errors.")

🗄️ DATABASE SCHEMA DIAGNOSTIC AND FIX
OPTION 1: Fix database schema
🗄️ FIXING DATABASE SCHEMA FOR ODDS
📊 CURRENT FIXTURE_ODDS TABLE SCHEMA:
   id                        | INTEGER    | Nullable: True
   fixture_id                | INTEGER    | Nullable: False
   bookmaker_id              | INTEGER    | Nullable: False
   bookmaker_name            | TEXT       | Nullable: True
   market_id                 | INTEGER    | Nullable: False
   market_name               | TEXT       | Nullable: True
   odds_label                | TEXT       | Nullable: True
   odds_value                | REAL       | Nullable: True
   probability               | REAL       | Nullable: True
   is_winning                | BOOLEAN    | Nullable: True
   updated_at                | TEXT       | Nullable: False

📈 Current records: 104,236,537

🔧 Attempting to add missing columns...
🔧 ADDING MISSING COLUMNS:
   ✅ Added: fractional (TEXT)
   ✅ Added: american (TEXT)
   ✅ Added: decimal_3 (TEXT)
   ✅ Added: handicap (

now we will once again test the second step again

it seems we already have the odds coverage inside the database? lets check it now again.

In [15]:
# Database Diagnostic - Find the Issue
# ===================================

import sqlite3
import pandas as pd

db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def full_database_diagnostic():
    """Comprehensive database diagnostic"""
    
    conn = sqlite3.connect(db_path)
    
    print("🔍 COMPREHENSIVE DATABASE DIAGNOSTIC")
    print("=" * 60)
    
    # 1. Check if fixture_odds table exists and structure
    print("1️⃣ FIXTURE_ODDS TABLE STRUCTURE:")
    try:
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(fixture_odds)")
        columns = cursor.fetchall()
        
        print(f"   Table exists with {len(columns)} columns:")
        for col in columns:
            print(f"      {col[1]:<25} | {col[2]:<10} | Nullable: {not col[3]}")
        
        # Check total count in fixture_odds
        cursor.execute("SELECT COUNT(*) FROM fixture_odds")
        total_odds = cursor.fetchone()[0]
        print(f"   📊 Total records in fixture_odds: {total_odds:,}")
        
    except Exception as e:
        print(f"   ❌ Error checking fixture_odds: {e}")
    
    # 2. Check constraints/indexes
    print(f"\n2️⃣ TABLE CONSTRAINTS:")
    try:
        cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='fixture_odds'")
        create_sql = cursor.fetchone()
        if create_sql:
            print(f"   Table creation SQL:")
            print(f"   {create_sql[0]}")
        
        # Check indexes
        cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='index' AND tbl_name='fixture_odds'")
        indexes = cursor.fetchall()
        if indexes:
            print(f"\n   Indexes on fixture_odds:")
            for idx in indexes:
                print(f"      {idx[0]}: {idx[1]}")
    except Exception as e:
        print(f"   ❌ Error checking constraints: {e}")
    
    # 3. Sample data from fixture_odds (if any)
    print(f"\n3️⃣ SAMPLE DATA FROM FIXTURE_ODDS:")
    try:
        sample_query = "SELECT * FROM fixture_odds LIMIT 5"
        sample_df = pd.read_sql_query(sample_query, conn)
        
        if len(sample_df) > 0:
            print("   Sample records found:")
            for col in sample_df.columns:
                print(f"      {col}: {sample_df[col].iloc[0] if len(sample_df) > 0 else 'NULL'}")
        else:
            print("   ⚠️ No records found in fixture_odds table")
            
    except Exception as e:
        print(f"   ❌ Error getting sample data: {e}")
    
    # 4. Check fixture_id values specifically
    print(f"\n4️⃣ CHECKING SPECIFIC FIXTURE IDS:")
    test_fixtures = [19354690, 19354697, 19347728, 19354665]
    
    for fixture_id in test_fixtures:
        try:
            cursor.execute("SELECT COUNT(*) FROM fixture_odds WHERE fixture_id = ?", (fixture_id,))
            count = cursor.fetchone()[0]
            print(f"   Fixture {fixture_id}: {count} records")
            
            if count > 0:
                # Get sample of this fixture's data
                cursor.execute("SELECT bookmaker_name, market_name, odds_label, odds_value FROM fixture_odds WHERE fixture_id = ? LIMIT 3", (fixture_id,))
                samples = cursor.fetchall()
                for i, sample in enumerate(samples):
                    print(f"      Sample {i+1}: {sample[0]} | {sample[1]} | {sample[2]} | {sample[3]}")
        except Exception as e:
            print(f"   ❌ Error checking fixture {fixture_id}: {e}")
    
    # 5. Check upcoming fixtures
    print(f"\n5️⃣ CHECKING UPCOMING FIXTURES:")
    try:
        upcoming_query = """
        SELECT COUNT(*) as total,
               COUNT(CASE WHEN starting_at > datetime('now') THEN 1 END) as future,
               MIN(starting_at) as earliest,
               MAX(starting_at) as latest
        FROM fixtures
        """
        upcoming = pd.read_sql_query(upcoming_query, conn)
        
        print(f"   Total fixtures: {upcoming.iloc[0]['total']:,}")
        print(f"   Future fixtures: {upcoming.iloc[0]['future']:,}")
        print(f"   Date range: {upcoming.iloc[0]['earliest']} to {upcoming.iloc[0]['latest']}")
        
    except Exception as e:
        print(f"   ❌ Error checking fixtures: {e}")
    
    # 6. Try a manual insert to see the exact error
    print(f"\n6️⃣ TESTING MANUAL INSERT:")
    try:
        test_insert = """
        INSERT INTO fixture_odds (
            id, fixture_id, market_id, bookmaker_id, 
            bookmaker_name, market_name, odds_label, odds_value
        ) VALUES (
            999999, 19354690, 1, 1, 
            'Test Bookmaker', 'Test Market', 'Test Label', 2.0
        )
        """
        
        cursor.execute(test_insert)
        print("   ✅ Test insert successful")
        
        # Remove the test record
        cursor.execute("DELETE FROM fixture_odds WHERE id = 999999")
        conn.commit()
        
    except Exception as e:
        print(f"   ❌ Test insert failed: {e}")
        print(f"      This explains the UNIQUE constraint error!")
    
    conn.close()
    
    print(f"\n🎯 DIAGNOSTIC COMPLETE")

def check_data_types():
    """Check if there are data type mismatches"""
    
    conn = sqlite3.connect(db_path)
    
    print(f"\n🔍 CHECKING DATA TYPES:")
    print("=" * 40)
    
    # Check what the current fixture_odds data looks like
    try:
        query = """
        SELECT 
            fixture_id,
            typeof(fixture_id) as fixture_id_type,
            market_id,
            typeof(market_id) as market_id_type,
            bookmaker_id,
            typeof(bookmaker_id) as bookmaker_id_type,
            odds_label,
            typeof(odds_label) as odds_label_type
        FROM fixture_odds 
        LIMIT 5
        """
        
        df = pd.read_sql_query(query, conn)
        
        if len(df) > 0:
            print("   Data types in existing records:")
            for _, row in df.iterrows():
                print(f"   fixture_id: {row['fixture_id']} ({row['fixture_id_type']})")
                print(f"   market_id: {row['market_id']} ({row['market_id_type']})")
                print(f"   bookmaker_id: {row['bookmaker_id']} ({row['bookmaker_id_type']})")
                print(f"   odds_label: '{row['odds_label']}' ({row['odds_label_type']})")
                break  # Just show first record
        else:
            print("   No existing records to check types")
            
    except Exception as e:
        print(f"   ❌ Error checking data types: {e}")
    
    conn.close()

# Run comprehensive diagnostic
if __name__ == "__main__":
    full_database_diagnostic()
    check_data_types()

🔍 COMPREHENSIVE DATABASE DIAGNOSTIC
1️⃣ FIXTURE_ODDS TABLE STRUCTURE:
   Table exists with 21 columns:
      id                        | INTEGER    | Nullable: True
      fixture_id                | INTEGER    | Nullable: False
      bookmaker_id              | INTEGER    | Nullable: False
      bookmaker_name            | TEXT       | Nullable: True
      market_id                 | INTEGER    | Nullable: False
      market_name               | TEXT       | Nullable: True
      odds_label                | TEXT       | Nullable: True
      odds_value                | REAL       | Nullable: True
      probability               | REAL       | Nullable: True
      is_winning                | BOOLEAN    | Nullable: True
      updated_at                | TEXT       | Nullable: False
      fractional                | TEXT       | Nullable: True
      american                  | TEXT       | Nullable: True
      decimal_3                 | TEXT       | Nullable: True
      handicap           

The UNIQUE constraint error persists, which means these specific fixtures DO have odds data in my database, but our query isn't finding them properly. Let me create a targeted check

In [19]:
# Targeted check for specific fixtures
# ===================================

import sqlite3
import pandas as pd

db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def check_specific_fixtures():
    """Check if these specific fixtures actually have odds"""
    
    conn = sqlite3.connect(db_path)
    test_fixtures = [19354690, 19354697, 19347728, 19354665]
    
    print("🔍 CHECKING SPECIFIC TEST FIXTURES")
    print("=" * 50)
    
    for fixture_id in test_fixtures:
        # Direct count
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM fixture_odds WHERE fixture_id = ?", (fixture_id,))
        direct_count = cursor.fetchone()[0]
        
        # Get fixture details
        cursor.execute("""
        SELECT f.starting_at, f.league_id, l.name as league, 
               ht.name as home_team, at.name as away_team
        FROM fixtures f
        JOIN leagues l ON f.league_id = l.id  
        JOIN teams ht ON f.home_team_id = ht.id
        JOIN teams at ON f.away_team_id = at.id
        WHERE f.id = ?
        """, (fixture_id,))
        
        fixture_info = cursor.fetchone()
        
        print(f"\n📋 FIXTURE {fixture_id}:")
        if fixture_info:
            print(f"   {fixture_info[3]} vs {fixture_info[4]}")
            print(f"   {fixture_info[2]} | {fixture_info[0]}")
            print(f"   📊 Odds count: {direct_count:,}")
            
            if direct_count > 0:
                # Show sample odds
                cursor.execute("""
                SELECT bookmaker_name, market_name, odds_label, odds_value
                FROM fixture_odds 
                WHERE fixture_id = ? 
                LIMIT 3
                """, (fixture_id,))
                
                samples = cursor.fetchall()
                print(f"   Sample odds:")
                for sample in samples:
                    print(f"      {sample[0]} | {sample[1]} | {sample[2]} | {sample[3]}")
        else:
            print(f"   ❌ Fixture not found in database")
    
    conn.close()

def find_fixtures_without_odds():
    """Find fixtures that truly don't have odds"""
    
    conn = sqlite3.connect(db_path)
    
    # More specific query 
    query = """
    SELECT 
        f.id as fixture_id,
        f.starting_at,
        l.name as league_name,
        ht.name as home_team,
        at.name as away_team,
        COALESCE(odds_count.count, 0) as existing_odds
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    JOIN teams ht ON f.home_team_id = ht.id
    JOIN teams at ON f.away_team_id = at.id
    LEFT JOIN (
        SELECT fixture_id, COUNT(*) as count
        FROM fixture_odds
        GROUP BY fixture_id
    ) odds_count ON f.id = odds_count.fixture_id
    WHERE f.starting_at > datetime('now')
    AND (odds_count.count IS NULL OR odds_count.count = 0)
    ORDER BY f.starting_at
    LIMIT 10
    """
    
    missing_odds = pd.read_sql_query(query, conn)
    
    print(f"\n🎯 FIXTURES TRULY WITHOUT ODDS:")
    print("=" * 50)
    
    if not missing_odds.empty:
        print(f"Found {len(missing_odds)} fixtures without odds:")
        for _, row in missing_odds.iterrows():
            print(f"   {row['fixture_id']} | {row['league_name']}")
            print(f"   {row['home_team']} vs {row['away_team']}")
            print(f"   {row['starting_at'][:16]} | Odds: {row['existing_odds']}")
            print()
    else:
        print("✅ ALL upcoming fixtures already have odds!")
    
    conn.close()
    return missing_odds

def check_overall_coverage():
    """Accurate coverage check"""
    
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT 
        COUNT(f.id) as total_upcoming,
        COUNT(CASE WHEN odds_count.count > 0 THEN 1 END) as with_odds,
        ROUND(COUNT(CASE WHEN odds_count.count > 0 THEN 1 END) * 100.0 / COUNT(f.id), 1) as coverage_pct,
        SUM(COALESCE(odds_count.count, 0)) as total_odds_upcoming
    FROM fixtures f
    LEFT JOIN (
        SELECT fixture_id, COUNT(*) as count
        FROM fixture_odds
        GROUP BY fixture_id
    ) odds_count ON f.id = odds_count.fixture_id
    WHERE f.starting_at > datetime('now')
    """
    
    coverage = pd.read_sql_query(query, conn)
    
    print(f"\n📊 ACCURATE COVERAGE STATISTICS:")
    print("=" * 40)
    print(f"Total upcoming fixtures: {coverage.iloc[0]['total_upcoming']:,}")
    print(f"Fixtures with odds: {coverage.iloc[0]['with_odds']:,}")
    print(f"Coverage: {coverage.iloc[0]['coverage_pct']}%")
    print(f"Total odds for upcoming: {coverage.iloc[0]['total_odds_upcoming']:,}")
    
    conn.close()
    return coverage.iloc[0].to_dict()

# Run the checks
if __name__ == "__main__":
    print("🎲 TARGETED ODDS INVESTIGATION")
    print("=" * 50)
    
    # Check the specific test fixtures
    check_specific_fixtures()
    
    # Find fixtures that truly need odds
    missing = find_fixtures_without_odds()
    
    # Get accurate coverage
    coverage = check_overall_coverage()
    
    print(f"\n🎯 CONCLUSION:")
    if coverage['coverage_pct'] >= 90:
        print("✅ EXCELLENT: Nearly all fixtures have odds!")
        print("✅ Ready for betting predictions!")
        print("✅ Skip odds fetching - proceed to model analysis!")
    elif len(missing) > 0:
        print(f"⚠️ {len(missing)} fixtures need odds fetching")
        print("💡 Use these fixture IDs for targeted fetching")
    else:
        print("🤔 Investigate further - data inconsistency detected")

🎲 TARGETED ODDS INVESTIGATION
🔍 CHECKING SPECIFIC TEST FIXTURES

📋 FIXTURE 19354690:
   Fredrikstad vs Rosenborg
   Eliteserien | 2025-05-28 17:00:00
   📊 Odds count: 0

📋 FIXTURE 19354697:
   Bodø / Glimt vs Viking
   Eliteserien | 2025-05-28 19:00:00
   📊 Odds count: 0

📋 FIXTURE 19347728:
   Brommapojkarna vs Djurgården
   Allsvenskan | 2025-05-29 16:00:00
   📊 Odds count: 0

📋 FIXTURE 19354665:
   Brann vs Molde
   Eliteserien | 2025-05-29 16:00:00
   📊 Odds count: 0

🎯 FIXTURES TRULY WITHOUT ODDS:
Found 10 fixtures without odds:
   19354690 | Eliteserien
   Fredrikstad vs Rosenborg
   2025-05-28 17:00 | Odds: 0

   19354697 | Eliteserien
   Bodø / Glimt vs Viking
   2025-05-28 19:00 | Odds: 0

   19419846 | Admiral Bundesliga
   TBC vs SK Rapid
   2025-05-29 15:00 | Odds: 0

   19347728 | Allsvenskan
   Brommapojkarna vs Djurgården
   2025-05-29 16:00 | Odds: 0

   19354665 | Eliteserien
   Brann vs Molde
   2025-05-29 16:00 | Odds: 0

   19376842 | Serie B
   Winner Semi-final 2 

this showcases, that all my odds in the database is purely historical, and we therefor are missing everything for the upcoming fixtures.

In [1]:
# Clean Working Odds Fetcher - Final Version
# ==========================================

import requests
import sqlite3
import pandas as pd
import time
from datetime import datetime
from tqdm import tqdm

# Configuration
API_TOKEN = "PgeMnb1Y71v04KzxFBpKQmm2sxsyWihIRNXSvDoYUz6ZuDOY3h1lLnmKamH1"
BASE_URL = "https://api.sportmonks.com/v3/football"
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def setup_session():
    """Setup API session"""
    session = requests.Session()
    session.headers.update({
        'Authorization': f'{API_TOKEN}',
        'Accept': 'application/json'
    })
    return session

def safe_float_convert(value, default=None):
    """Safely convert value to float"""
    if value is None or value == '':
        return default
    try:
        if isinstance(value, str) and '%' in value:
            return float(value.replace('%', '')) / 100.0
        return float(value)
    except (ValueError, TypeError):
        return default

def safe_int_convert(value, default=0):
    """Safely convert value to int"""
    if value is None or value == '':
        return default
    try:
        return int(float(value))  # Handle cases like "1.0"
    except (ValueError, TypeError):
        return default

def get_fixtures_needing_odds_clean():
    """Get fixtures that truly need odds"""
    
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT 
        f.id as fixture_id,
        f.starting_at,
        l.name as league_name,
        ht.name as home_team,
        at.name as away_team
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    JOIN teams ht ON f.home_team_id = ht.id
    JOIN teams at ON f.away_team_id = at.id
    WHERE f.starting_at > datetime('now')
    AND f.id NOT IN (
        SELECT DISTINCT fixture_id 
        FROM fixture_odds 
        WHERE fixture_id IS NOT NULL
    )
    ORDER BY f.starting_at
    """
    
    fixtures_df = pd.read_sql_query(query, conn)
    conn.close()
    
    print(f"📊 Found {len(fixtures_df)} fixtures needing odds")
    return fixtures_df

def fetch_fixture_odds_clean(session, fixture_id):
    """Fetch odds for a fixture"""
    
    url = f"{BASE_URL}/odds/pre-match/fixtures/{fixture_id}"
    
    try:
        response = session.get(url, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            return data.get('data', [])
        elif response.status_code == 429:
            print(f"      ⏳ Rate limited, waiting...")
            time.sleep(60)
            return None  # Signal to retry
        elif response.status_code == 404:
            return []  # No odds available
        else:
            print(f"      ❌ Error {response.status_code}")
            return []
            
    except Exception as e:
        print(f"      ❌ Request failed: {str(e)[:100]}")
        return []

def process_odds_clean(odds_data, fixture_id):
    """Process odds data for insertion"""
    
    processed = []
    current_time = datetime.now().isoformat()
    
    for odds in odds_data:
        processed.append({
            'fixture_id': safe_int_convert(fixture_id),
            'market_id': safe_int_convert(odds.get('market_id')),
            'bookmaker_id': safe_int_convert(odds.get('bookmaker_id')),
            'bookmaker_name': str(odds.get('name', 'Unknown')),
            'market_name': str(odds.get('market_description', 'Unknown')),
            'odds_label': str(odds.get('label', '')),
            'odds_value': safe_float_convert(odds.get('value')),
            'probability': safe_float_convert(odds.get('probability')),
            'fractional': str(odds.get('fractional', '')),
            'american': str(odds.get('american', '')),
            'decimal_3': str(odds.get('dp3', '')),
            'handicap': safe_float_convert(odds.get('handicap')),
            'total': safe_float_convert(odds.get('total')),
            'sort_order': safe_int_convert(odds.get('sort_order')),
            'winning': odds.get('winning'),
            'stopped': bool(odds.get('stopped', False)),
            'participants': str(odds.get('participants', '')),
            'latest_bookmaker_update': str(odds.get('latest_bookmaker_update', '')),
            'is_winning': None,
            'updated_at': current_time
        })
    
    return processed

def insert_odds_clean(odds_batch):
    """Clean insertion with proper error handling"""
    
    if not odds_batch:
        return 0
    
    conn = sqlite3.connect(db_path)
    
    try:
        # Use executemany for better control
        insert_sql = """
        INSERT OR IGNORE INTO fixture_odds (
            fixture_id, market_id, bookmaker_id, bookmaker_name, market_name,
            odds_label, odds_value, probability, fractional, american, decimal_3,
            handicap, total, sort_order, winning, stopped, participants,
            latest_bookmaker_update, is_winning, updated_at
        ) VALUES (
            ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
        )
        """
        
        cursor = conn.cursor()
        
        # Prepare data for insertion
        insert_data = []
        for odds in odds_batch:
            insert_data.append((
                odds['fixture_id'], odds['market_id'], odds['bookmaker_id'],
                odds['bookmaker_name'], odds['market_name'], odds['odds_label'],
                odds['odds_value'], odds['probability'], odds['fractional'],
                odds['american'], odds['decimal_3'], odds['handicap'],
                odds['total'], odds['sort_order'], odds['winning'],
                odds['stopped'], odds['participants'], odds['latest_bookmaker_update'],
                odds['is_winning'], odds['updated_at']
            ))
        
        cursor.executemany(insert_sql, insert_data)
        conn.commit()
        
        # Count actual insertions (not ignored due to duplicates)
        inserted = cursor.rowcount
        conn.close()
        
        return inserted
        
    except Exception as e:
        print(f"      ❌ Database error: {e}")
        conn.rollback()
        conn.close()
        return 0

def fetch_all_odds_clean(test_mode=False, limit=5):
    """Main function - clean odds fetching"""
    
    print("🎲 CLEAN ODDS FETCHING")
    print("=" * 40)
    
    # Get fixtures
    fixtures_df = get_fixtures_needing_odds_clean()
    
    if len(fixtures_df) == 0:
        print("✅ All fixtures already have odds!")
        return {'successful': 0, 'failed': 0, 'total_odds': 0}
    
    if test_mode:
        fixtures_df = fixtures_df.head(limit)
        print(f"🧪 TEST MODE: Processing {len(fixtures_df)} fixtures")
    
    # Setup
    session = setup_session()
    stats = {'successful': 0, 'failed': 0, 'total_odds': 0}
    
    # Process each fixture
    for i, (_, fixture) in enumerate(fixtures_df.iterrows(), 1):
        fixture_id = fixture['fixture_id']
        match_info = f"{fixture['home_team']} vs {fixture['away_team']}"
        
        print(f"\n{i}/{len(fixtures_df)} | {fixture['league_name']}")
        print(f"   {match_info}")
        
        # Fetch odds
        odds_data = fetch_fixture_odds_clean(session, fixture_id)
        
        # Handle rate limiting
        if odds_data is None:  # Rate limited
            time.sleep(60)
            odds_data = fetch_fixture_odds_clean(session, fixture_id)
        
        if odds_data and len(odds_data) > 0:
            # Process and insert
            processed_odds = process_odds_clean(odds_data, fixture_id)
            inserted = insert_odds_clean(processed_odds)
            
            stats['successful'] += 1
            stats['total_odds'] += inserted
            
            print(f"   ✅ {len(processed_odds):,} odds fetched, {inserted:,} inserted")
            
        else:
            stats['failed'] += 1
            print(f"   ❌ No odds available")
        
        # Rate limiting
        time.sleep(1.2)
    
    # Summary
    print(f"\n🎉 ODDS FETCHING COMPLETE!")
    print("=" * 30)
    print(f"✅ Successful: {stats['successful']}")
    print(f"❌ Failed: {stats['failed']}")
    print(f"📊 Total odds: {stats['total_odds']:,}")
    
    return stats

def verify_odds_insertion():
    """Verify odds were actually inserted"""
    
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT 
        COUNT(DISTINCT f.id) as total_upcoming,
        COUNT(DISTINCT fo.fixture_id) as with_odds,
        ROUND(COUNT(DISTINCT fo.fixture_id) * 100.0 / COUNT(DISTINCT f.id), 1) as coverage,
        COUNT(*) as total_odds
    FROM fixtures f
    LEFT JOIN fixture_odds fo ON f.id = fo.fixture_id
    WHERE f.starting_at > datetime('now')
    """
    
    result = pd.read_sql_query(query, conn)
    conn.close()
    
    print(f"\n📊 VERIFICATION RESULTS:")
    print("=" * 30)
    print(f"Total upcoming fixtures: {result.iloc[0]['total_upcoming']:,}")
    print(f"Fixtures with odds: {result.iloc[0]['with_odds']:,}")
    print(f"Coverage: {result.iloc[0]['coverage']}%")
    print(f"Total odds records: {result.iloc[0]['total_odds']:,}")
    
    return result.iloc[0].to_dict()

# Test and full functions
def test_clean_odds():
    """Test with 5 fixtures"""
    return fetch_all_odds_clean(test_mode=True, limit=5)

def fetch_all_clean():
    """Fetch all odds"""
    return fetch_all_odds_clean(test_mode=False)

# Usage
if __name__ == "__main__":
    print("🎲 CLEAN ODDS FETCHER")
    print("=" * 30)
    print("Functions available:")
    print("1. test_clean_odds()    - Test with 5 fixtures")
    print("2. fetch_all_clean()    - Fetch all odds")
    print("3. verify_odds_insertion() - Check results")

🎲 CLEAN ODDS FETCHER
Functions available:
1. test_clean_odds()    - Test with 5 fixtures
2. fetch_all_clean()    - Fetch all odds
3. verify_odds_insertion() - Check results


In [2]:
# Test the clean version with fixed data type handling
print("🧪 TESTING CLEAN ODDS FETCHER (FIXED)")
test_results = test_clean_odds()

🧪 TESTING CLEAN ODDS FETCHER (FIXED)
🎲 CLEAN ODDS FETCHING
📊 Found 348 fixtures needing odds
🧪 TEST MODE: Processing 5 fixtures

1/5 | Eliteserien
   Fredrikstad vs Rosenborg
   ✅ 2,752 odds fetched, 1,446 inserted

2/5 | Eliteserien
   Bodø / Glimt vs Viking
   ✅ 2,967 odds fetched, 1,486 inserted

3/5 | Admiral Bundesliga
   TBC vs SK Rapid
   ❌ No odds available

4/5 | Allsvenskan
   Brommapojkarna vs Djurgården
   ✅ 2,801 odds fetched, 1,404 inserted

5/5 | Eliteserien
   Brann vs Molde
   ✅ 2,877 odds fetched, 1,458 inserted

🎉 ODDS FETCHING COMPLETE!
✅ Successful: 4
❌ Failed: 1
📊 Total odds: 5,794


In [3]:
# Fetch odds for all 348 fixtures
print("🚀 FETCHING ALL ODDS DATA")
print("This will take approximately 15-20 minutes...")
print("Processing 348 fixtures with rate limiting...")

full_results = fetch_all_clean()

# Verify the results
print("\n🔍 VERIFYING FINAL RESULTS...")
final_status = verify_odds_insertion()

🚀 FETCHING ALL ODDS DATA
This will take approximately 15-20 minutes...
Processing 348 fixtures with rate limiting...
🎲 CLEAN ODDS FETCHING
📊 Found 344 fixtures needing odds

1/344 | Admiral Bundesliga
   TBC vs SK Rapid
   ❌ No odds available

2/344 | Serie B
   Winner Semi-final 2 vs Winner Semi-final 1
   ✅ 2,479 odds fetched, 1,362 inserted

3/344 | Allsvenskan
   Elfsborg vs Hammarby
   ✅ 2,467 odds fetched, 1,233 inserted

4/344 | Allsvenskan
   Norrköping vs GAIS
   ✅ 2,470 odds fetched, 1,223 inserted

5/344 | Allsvenskan
   Degerfors vs Öster
   ✅ 2,406 odds fetched, 1,229 inserted

6/344 | Eliteserien
   Strømsgodset vs HamKam
   ✅ 1,721 odds fetched, 1,095 inserted

7/344 | Eliteserien
   Tromsø vs Vålerenga
   ✅ 2,368 odds fetched, 1,220 inserted

8/344 | Superliga
   TBC vs TBC
   ❌ No odds available

9/344 | La Liga 2
   Almería vs Tenerife
   ✅ 3,090 odds fetched, 1,307 inserted

10/344 | La Liga 2
   FC Cartagena vs Mirandés
   ✅ 2,562 odds fetched, 1,274 inserted

11/34